In [29]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Helper Functions (Signal Loading)
# ==========================================
def create_ticker_to_permno_mapping(df):
    valid_df = df[df['ticker'].notna()].copy()
    return valid_df.groupby('ticker')['permno'].last().to_dict()

def load_yearly_signals(year, buys_temp, sells_temp):
    try:
        buys = pd.read_csv(buys_temp.format(year), index_col=1)
        sells = pd.read_csv(sells_temp.format(year), index_col=1)
        return set(buys.index.astype(int).union(sells.index.astype(int)))
    except FileNotFoundError:
        return set()

def load_finbert_signals(path):
    try:
        df = pd.read_csv(path)
        df['date'] = pd.to_datetime(df['year_month']) + pd.offsets.MonthEnd(0)
        return df
    except FileNotFoundError:
        return pd.DataFrame()

def get_finbert_permnos(signals_df, ticker_map, date):
    date_signals = signals_df[signals_df['date'] == date]
    active_signals = date_signals[date_signals['signal'].isin(['buy', 'sell'])]
    
    permnos = set()
    for ticker in active_signals['symbol'].values:
        if ticker in ticker_map:
            permnos.add(ticker_map[ticker])
    return permnos

# --- UPDATED: 15-Year Continuous Data Filter & 1-Stock Rule Added ---
def build_screening_dictionary(test_dates, all_dates, full_pivot, df_universe, buys_temp, sells_temp, finbert_df, lookback_window=180):
    screened_dict = {}
    valid_universe_dict = {}
    all_stocks_dict = {}  # NEW: Tracks every stock available in the current month
    
    ticker_map = create_ticker_to_permno_mapping(df_universe)
    yearly_cache = {}
    
    for date in test_dates:
        year = date.year
        
        # --- NEW: Track all stocks available in this specific month ---
        current_month_permnos = set(df_universe[df_universe['datadate'] == date]['permno'])
        all_stocks_dict[date] = current_month_permnos
        
        # 1. Enforce 15-year (180 months) continuous data requirement
        try:
            t = all_dates.index(date)
            if t < lookback_window:
                print(f"Not enough history for {date.strftime('%Y-%m')}. Skipping.")
                continue
            
            # Slice the 180 months strictly prior to the current test date
            window_data = full_pivot.iloc[t - lookback_window : t]
            # Keep permnos that have 0 NaNs in this 180-month window
            valid_permnos = set(window_data.columns[window_data.notna().all()])
            valid_universe_dict[date] = valid_permnos
            
        except ValueError:
            continue
            
        # 2. Get Signals
        if year not in yearly_cache:
            yearly_cache[year] = load_yearly_signals(year, buys_temp, sells_temp)
            
        yearly_permnos = yearly_cache[year]
        finbert_permnos = get_finbert_permnos(finbert_df, ticker_map, date) if not finbert_df.empty else set()
        
        # Intersection fallback to Union
        allowed = yearly_permnos.intersection(finbert_permnos)
        if len(allowed) <= 1:
            allowed = yearly_permnos.union(finbert_permnos)
            
        # 3. Final Screened Set: Signals INTERSECT 15-Year Continuous Stocks
        final_screened = allowed.intersection(valid_permnos)
        
        # If only 1 stock, treat as 0
        if len(final_screened) == 1:
            final_screened = set() 
            
        screened_dict[date] = final_screened
        
    return screened_dict, valid_universe_dict, all_stocks_dict

# ==========================================
# 2. Variance Calculation Function
# ==========================================
# ==========================================
# 2. Variance Calculation Function
# ==========================================
def calculate_variance_metrics(df, screened_dict, valid_universe_dict, all_stocks_dict, ret_col='ret_fwd_1'):
    results = []
    
    for date, group in df.groupby('datadate'):
        if date not in screened_dict or len(screened_dict[date]) == 0:
            continue
            
        allowed_permnos = screened_dict[date]
        universe_permnos = valid_universe_dict[date]
        all_permnos = all_stocks_dict[date]
        
        # Isolate returns for all three tiers
        all_rets = group[group['permno'].isin(all_permnos)][ret_col].dropna()
        univ_rets = group[group['permno'].isin(universe_permnos)][ret_col].dropna()
        screened_rets = group[group['permno'].isin(allowed_permnos)][ret_col].dropna()
        
        if len(univ_rets) < 2 or len(screened_rets) < 2:
            continue
            
        # Metric components
        results.append({
            'date': date,
            'all_xs_var': all_rets.var(ddof=1) if len(all_rets) >= 2 else np.nan,
            'univ_xs_var': univ_rets.var(ddof=1),
            'screened_xs_var': screened_rets.var(ddof=1),
            'all_ew_ret': all_rets.mean(),
            'univ_ew_ret': univ_rets.mean(),
            'screened_ew_ret': screened_rets.mean(),
            'all_n': len(all_rets),
            'univ_n': len(univ_rets),
            'screened_n': len(screened_rets)
        })
        
    res_df = pd.DataFrame(results)
    
    if not res_df.empty:
        # (i) Average Cross-Sectional Variance
        avg_all_xs = res_df['all_xs_var'].mean() * 12
        avg_univ_xs = res_df['univ_xs_var'].mean() * 12
        avg_screened_xs = res_df['screened_xs_var'].mean() * 12
        
        # (ii) Time-Series Variance of EW Portfolios
        ts_all_ew = res_df['all_ew_ret'].var(ddof=1) * 12
        ts_univ_ew = res_df['univ_ew_ret'].var(ddof=1) * 12
        ts_screened_ew = res_df['screened_ew_ret'].var(ddof=1) * 12
        
        print("\n" + "="*70)
        print(f"VARIANCE ANALYSIS: {res_df['date'].min().strftime('%Y-%m')} to {res_df['date'].max().strftime('%Y-%m')}")
        print("="*70)
        print("\n(i) Average Cross-Sectional Return Variance")
        print(f"  All Stocks           : {avg_all_xs:.6f}")
        print(f"  Valid Universe (15-yr hist): {avg_univ_xs:.6f}")
        print(f"  Screened Set               : {avg_screened_xs:.6f}")
        
        print("\n(ii) Time-Series Variance of Equal-Weighted Portfolios")
        print(f"  All Stocks EW        : {ts_all_ew:.6f}")
        print(f"  Valid Univ EW (15-yr hist) : {ts_univ_ew:.6f}")
        print(f"  Screened EW                : {ts_screened_ew:.6f}")
        print("="*70)
        
    return res_df

In [30]:
# ==========================================
# 3. Main Execution Block
# ==========================================
print("Loading data...")

# Load Green dataset
df = pd.read_csv('../green cleaned.csv', dtype={'ncusip': 'string'})
df['datadate'] = pd.to_datetime(df['datadate'])
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

all_dates = sorted(df['datadate'].unique())

# Pivot full historical returns once to speed up the 15-year lookback check
print("Pivoting return history for 15-year continuity checks...")
full_pivot = df.pivot(index='datadate', columns='permno', values='ret_fwd_1')

# Define test period
test_start_date = pd.to_datetime('2021-10-31')
test_end_date = pd.to_datetime('2024-04-30')

test_df = df[(df['datadate'] >= test_start_date) & (df['datadate'] <= test_end_date)].copy()
test_dates = sorted(test_df['datadate'].unique())

Loading data...
Pivoting return history for 15-year continuity checks...


In [31]:
print(f"Analyzing {len(test_dates)} months...")

# Load FinBERT signals
finbert_data = load_finbert_signals('../examples/monthly_signals_decay.csv')

# Build the screening dictionary enforcing the 180-month (15-year) continuous rule
screened_dict, valid_universe_dict, all_stocks_dict = build_screening_dictionary(
    test_dates=test_dates, 
    all_dates=all_dates,
    full_pivot=full_pivot,
    df_universe=df, 
    buys_temp='../GPT 3.5/Run 1 (Main)/buys_{}.csv', 
    sells_temp='../GPT 3.5/Run 1 (Main)/sells_{}.csv',
    finbert_df=finbert_data,
    lookback_window=180
)

# Run the variance calculations with the new 3-tier inputs
variance_results_df = calculate_variance_metrics(
    df=test_df, 
    screened_dict=screened_dict, 
    valid_universe_dict=valid_universe_dict, 
    all_stocks_dict=all_stocks_dict, 
    ret_col='ret_fwd_1'
)

Analyzing 31 months...

VARIANCE ANALYSIS: 2021-10 to 2024-04

(i) Average Cross-Sectional Return Variance
  All Stocks           : 0.071529
  Valid Universe (15-yr hist): 0.061690
  Screened Set               : 0.153732

(ii) Time-Series Variance of Equal-Weighted Portfolios
  All Stocks EW        : 0.040184
  Valid Univ EW (15-yr hist) : 0.035500
  Screened EW                : 0.123725


In [32]:
# ==========================================
# Print Screened Universe Size Over Time
# ==========================================
print("\n" + "="*40)
print("SCREENED PORTFOLIO SIZE OVER TIME")
print("="*40)

counts_list = []
for d in sorted(screened_dict.keys()):
    counts_list.append({
        'Date': d.strftime('%Y-%m'), 
        'Total_Screened_Stocks': len(screened_dict[d])
    })

counts_df = pd.DataFrame(counts_list)
print(counts_df.to_string(index=False))
print("="*40)


SCREENED PORTFOLIO SIZE OVER TIME
   Date  Total_Screened_Stocks
2021-10                     23
2021-11                     18
2021-12                     19
2022-01                     22
2022-02                     17
2022-03                     19
2022-04                     21
2022-05                     13
2022-06                     13
2022-07                     22
2022-08                     16
2022-09                     19
2022-10                     23
2022-11                     23
2022-12                     15
2023-01                      0
2023-02                      3
2023-03                      0
2023-04                      0
2023-05                      2
2023-06                      4
2023-07                      3
2023-08                      0
2023-09                      0
2023-10                      2
2023-11                      3
2023-12                      5
2024-01                     30
2024-02                     22
2024-03                     18
2024

We verified this is the exact sequence of sizes that happens for LLM-S + FINBERT.

Metric (i): Average Cross-Sectional Return VarianceWhat it measures: How "dispersed" or spread out the individual stock returns are from each other in a typical month.If this number is high, it means that in any given month, some stocks in your portfolio are soaring while others are crashing. If it is low, the stocks are moving relatively tightly together.How it is calculated (Step-by-Step):For a single month ($t$): We look at all $N_t$ stocks in the portfolio for that specific month. We calculate the cross-sectional sample variance of their returns.$$\sigma^2_t = \frac{1}{N_t - 1} \sum_{i=1}^{N_t} (R_{i,t} - \bar{R}_t)^2$$(Where $R_{i,t}$ is the return of stock $i$ in month $t$, and $\bar{R}_t$ is the cross-sectional average return of the portfolio in month $t$.)In code: This happens in the loop via screened_rets.var(ddof=1).Over the whole period ($T$): We take those monthly cross-sectional variances and calculate their simple time-series average.$$\text{Metric (i)} = \frac{1}{T} \sum_{t=1}^{T} \sigma^2_t$$In code: This happens at the end via res_df['screened_xs_var'].mean().

Metric (ii): Time-Series Variance of Equal-Weighted PortfoliosWhat it measures: The standard volatility (risk) of the portfolio as a whole over time.If this number is high, the overall portfolio's return jumps around drastically from one month to the next.How it is calculated (Step-by-Step):For a single month ($t$): We calculate the return of the equal-weighted portfolio for that month (just the simple average of all stock returns in the portfolio).$$P_t = \frac{1}{N_t} \sum_{i=1}^{N_t} R_{i,t}$$In code: This happens in the loop via screened_rets.mean().Over the whole period ($T$): This gives us a single time-series of monthly portfolio returns: $[P_1, P_2, \dots, P_T]$. We then calculate the standard time-series sample variance of this array.$$\text{Metric (ii)} = \frac{1}{T - 1} \sum_{t=1}^{T} (P_t - \mu_P)^2$$(Where $\mu_P$ is the overall average portfolio return across all $T$ months).In code: This happens at the end via res_df['screened_ew_ret'].var(ddof=1).